In [1]:
# ============================================================
#  Emotion Classifier — DistilBERT Fine-tuning
#  pip install transformers datasets scikit-learn torch pandas
# ============================================================
#  Run this in Colab with GPU:
#  Runtime → Change runtime type → T4 GPU
# ============================================================

import pandas as pd
import pickle
import numpy as np
import torch
from datasets import load_dataset, Dataset
from transformers import (
    DistilBertTokenizerFast,
    DistilBertForSequenceClassification,
    TrainingArguments,
    Trainer,
)
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report
from sklearn.preprocessing import LabelEncoder




In [2]:
#Getting the go-emotions file from my computer
from google.colab import files
uploaded = files.upload()  # a file picker will appear

Saving archive.zip to archive.zip


In [3]:
#unzipping the file
import zipfile
with zipfile.ZipFile("archive.zip", "r") as z:
    z.extractall("goemotions")

import os
print(os.listdir("goemotions"))  # see what files are inside
print(os.listdir("goemotions/data"))

['data', 'extract_words.py', 'tables', 'GoEmotionsFormat.PNG', 'plots', 'calculate_metrics.py', 'goemotions_model_card.pdf', 'replace_emotions.py', 'README.md', 'analyze_data.py']
['full_dataset', 'ekman_labels.csv', 'sentiment_mapping.json', 'sentiment_dict.json', 'train.tsv', 'emotions.txt', 'dev.tsv', 'test.tsv', 'ekman_mapping.json']


In [4]:
#============================================================
# STEP 1 — Build dataset (same as before)
# ============================================================

print("Loading dair-ai/emotion...")
hf_ds = load_dataset("dair-ai/emotion", split="train")
hf_df = pd.DataFrame({"text": hf_ds["text"], "label": hf_ds["label"]})
hf_map = {0:"sad", 1:"happy", 2:"excited", 3:"angry", 4:"fear", 5:"excited"}
hf_df["emotion"] = hf_df["label"].map(hf_map)
hf_df = hf_df.dropna(subset=["emotion"])[["text","emotion"]]
print("Finished dair air database")

Loading dair-ai/emotion...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


README.md: 0.00B [00:00, ?B/s]

split/train-00000-of-00001.parquet:   0%|          | 0.00/1.03M [00:00<?, ?B/s]

split/validation-00000-of-00001.parquet:   0%|          | 0.00/127k [00:00<?, ?B/s]

split/test-00000-of-00001.parquet:   0%|          | 0.00/129k [00:00<?, ?B/s]

Generating train split:   0%|          | 0/16000 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/2000 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/2000 [00:00<?, ? examples/s]

Finished dair air database


In [21]:
# --- Local GoEmotions TSV (uploaded from Kaggle) ---
print("Loading go_emotions from local file...")
with open("goemotions/data/emotions.txt") as f:
    emotion_labels = [line.strip() for line in f.readlines()]

confusion_idx   = emotion_labels.index("confusion")
curiosity_idx   = emotion_labels.index("curiosity")
admiration_idx  = emotion_labels.index("admiration")
excitement_idx  = emotion_labels.index("excitement")
nervousness_idx = emotion_labels.index("nervousness")
annoyance_idx   = emotion_labels.index("annoyance")
realization_idx = emotion_labels.index("realization")

go_raw = pd.read_csv("goemotions/data/train.tsv", sep="\t",
                     header=None, names=["text","labels","id"])
# Also pull dev and test TSVs for more confused samples
go_files = ["goemotions/data/train.tsv",
            "goemotions/data/dev.tsv",
            "goemotions/data/test.tsv"]

go_rows = []
for filepath in go_files:
    go_raw = pd.read_csv(filepath, sep="\t",
                         header=None, names=["text","labels","id"])
    for _, row in go_raw.iterrows():
        idxs = [int(x) for x in str(row["labels"]).split(",")]
        if confusion_idx in idxs:
            go_rows.append({"text": row["text"], "emotion": "confused"})
        elif curiosity_idx in idxs or realization_idx in idxs:
            go_rows.append({"text": row["text"], "emotion": "thinking"})
        elif excitement_idx in idxs or admiration_idx in idxs:
            go_rows.append({"text": row["text"], "emotion": "excited"})
        elif nervousness_idx in idxs:
            go_rows.append({"text": row["text"], "emotion": "fear"})
        elif annoyance_idx in idxs:
            go_rows.append({"text": row["text"], "emotion": "angry"})

print("Loading neutral sentences...")
neutral_ds = load_dataset("mteb/tweet_sentiment_extraction", split="train")
neutral_rows = []
for row in neutral_ds:
    if row["label_text"] == "neutral" and len(row["text"].strip()) > 10:
        neutral_rows.append({"text": row["text"], "emotion": "neutral"})
    if len(neutral_rows) >= 4000: break
neutral_df = pd.DataFrame(neutral_rows)

Loading go_emotions from local file...
Loading neutral sentences...


In [23]:

#Merge, clean & balance
df = pd.concat([hf_df, go_df, neutral_df], ignore_index=True)
df = df.dropna().drop_duplicates(subset=["text"])

MIN_SAMPLES = 2500
balanced = (
    df.groupby("emotion", group_keys=False)
    .apply(lambda x: x.sample(min(len(x), MIN_SAMPLES), random_state=42))
    .reset_index(drop=True)
)
print("\nBalanced class distribution:")
print(balanced["emotion"].value_counts())


Balanced class distribution:
emotion
angry       2500
excited     2500
neutral     2500
happy       2500
sad         2500
thinking    2500
fear        2088
confused    1364
Name: count, dtype: int64


/tmp/ipykernel_776/1601418448.py:8: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(lambda x: x.sample(min(len(x), MIN_SAMPLES), random_state=42))


In [24]:

# ============================================================
# STEP 2 — Encode labels
# ============================================================
le = LabelEncoder()
balanced["label_id"] = le.fit_transform(balanced["emotion"])
print("\nLabel mapping:", dict(zip(le.classes_, le.transform(le.classes_))))

X_train, X_test, y_train, y_test = train_test_split(
    balanced["text"].tolist(),
    balanced["label_id"].tolist(),
    test_size=0.2, random_state=42, stratify=balanced["label_id"]
)



Label mapping: {'angry': np.int64(0), 'confused': np.int64(1), 'excited': np.int64(2), 'fear': np.int64(3), 'happy': np.int64(4), 'neutral': np.int64(5), 'sad': np.int64(6), 'thinking': np.int64(7)}


In [25]:

# ============================================================
# STEP 3 — Tokenize
# ============================================================
MODEL_NAME = "distilbert-base-uncased"
tokenizer = DistilBertTokenizerFast.from_pretrained(MODEL_NAME)

def tokenize(texts, labels):
    enc = tokenizer(texts, truncation=True, padding=True, max_length=128)
    enc["labels"] = labels
    return Dataset.from_dict(enc)

train_ds = tokenize(X_train, y_train)
test_ds  = tokenize(X_test,  y_test)


In [26]:

# ============================================================
# STEP 4 — Load DistilBERT
# ============================================================
num_labels = len(le.classes_)
model = DistilBertForSequenceClassification.from_pretrained(
    MODEL_NAME, num_labels=num_labels
)


Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_layer_norm.weight | UNEXPECTED | 
vocab_transform.weight  | UNEXPECTED | 
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_projector.bias    | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
pre_classifier.bias     | MISSING    | 
classifier.weight       | MISSING    | 
pre_classifier.weight   | MISSING    | 
classifier.bias         | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


In [27]:
# ============================================================
# STEP 5 — Training arguments
# ============================================================
args = TrainingArguments(
    output_dir="./emotion_bert",
    num_train_epochs=4,
    per_device_train_batch_size=32,
    per_device_eval_batch_size=64,
    warmup_steps=200,
    weight_decay=0.01,
    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    logging_steps=50,
    fp16=torch.cuda.is_available(),  # faster training on GPU
)

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=1)
    acc = (preds == labels).mean()
    return {"accuracy": acc}

trainer = Trainer(
    model=model,
    args=args,
    train_dataset=train_ds,
    eval_dataset=test_ds,
    compute_metrics=compute_metrics,
)


In [28]:

# ============================================================
# STEP 6 — Train
# ============================================================
print("\nTraining DistilBERT...")
trainer.train()




Training DistilBERT...


Epoch,Training Loss,Validation Loss,Accuracy
1,0.578052,0.525978,0.811162
2,0.397356,0.447900,0.835546
3,0.233956,0.452469,0.848821
4,0.136678,0.469493,0.849905


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

There were missing keys in the checkpoint model loaded: ['distilbert.embeddings.LayerNorm.weight', 'distilbert.embeddings.LayerNorm.bias'].
There were unexpected keys in the checkpoint model loaded: ['distilbert.embeddings.LayerNorm.beta', 'distilbert.embeddings.LayerNorm.gamma'].


TrainOutput(global_step=1848, training_loss=0.4590996395457875, metrics={'train_runtime': 230.069, 'train_samples_per_second': 256.636, 'train_steps_per_second': 8.032, 'total_flos': 1680559802148480.0, 'train_loss': 0.4590996395457875, 'epoch': 4.0})

In [29]:
# ============================================================
# STEP 7 — Evaluate
# ============================================================
preds_output = trainer.predict(test_ds)
y_pred = np.argmax(preds_output.predictions, axis=1)
print("\nClassification Report:")
print(classification_report(y_test, y_pred, target_names=le.classes_))



Classification Report:
              precision    recall  f1-score   support

       angry       0.81      0.77      0.79       500
    confused       0.71      0.43      0.54       273
     excited       0.88      0.80      0.84       500
        fear       0.92      0.92      0.92       418
       happy       0.94      0.93      0.93       500
     neutral       0.96      0.87      0.91       500
         sad       0.92      0.98      0.95       500
    thinking       0.59      0.82      0.68       500

    accuracy                           0.84      3691
   macro avg       0.84      0.81      0.82      3691
weighted avg       0.85      0.84      0.84      3691



In [30]:

# ============================================================
# STEP 8 — Save as pickle
# ============================================================
# We save the model, tokenizer, and label encoder together
# so emotion_server.py can load everything from one file
bundle = {
    "model_dir": "./emotion_bert_final",
    "label_encoder": le,
}
model.save_pretrained("./emotion_bert_final")
tokenizer.save_pretrained("./emotion_bert_final")

with open("emotion_model.pkl", "wb") as f:
    pickle.dump(bundle, f)

print("\n✅ Model saved to emotion_model.pkl + ./emotion_bert_final/")


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]


✅ Model saved to emotion_model.pkl + ./emotion_bert_final/


In [34]:

# ============================================================
# STEP 9 — Sanity check
# ============================================================
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)

def predict(text):
    inputs = tokenizer(text, return_tensors="pt",
                       truncation=True, padding=True, max_length=128)
    inputs = {k: v.to(device) for k, v in inputs.items()}  # move to GPU
    with torch.no_grad():
        logits = model(**inputs).logits
    pred_id = torch.argmax(logits, dim=1).item()
    return le.inverse_transform([pred_id])[0]

test_sentences = [
    ("I just got the best news of my life!", "happy"),
    ("I have no idea what's going on here.", "confused"),
    ("I can't wait for the concert tonight!!", "excited"),
    ("I miss them so much, it hurts.", "sad"),
    ("The package arrived on Tuesday.", "neutral"),
    ("Hmm, I wonder why that happens...", "thinking"),
    ("I am so furious right now!", "angry"),
    ("I'm scared of what might happen next.", "fear"),
    ("I'm a bit tired.", "neutral"),
    ("I'm can't wait for my birthday", "excited"),
    ("How long does it take to pick up up a piece of trash", "angry"),
    ("tomato tomato", "neutral"),
]
print("\nSanity check:")
for text, expected in test_sentences:
    pred = predict(text)
    status = "✅" if pred == expected else "❌"
    print(f"  {status} '{text[:45]}' → {pred} (expected {expected})")


Sanity check:
  ❌ 'I just got the best news of my life!' → excited (expected happy)
  ✅ 'I have no idea what's going on here.' → confused (expected confused)
  ✅ 'I can't wait for the concert tonight!!' → excited (expected excited)
  ❌ 'I miss them so much, it hurts.' → neutral (expected sad)
  ✅ 'The package arrived on Tuesday.' → neutral (expected neutral)
  ✅ 'Hmm, I wonder why that happens...' → thinking (expected thinking)
  ✅ 'I am so furious right now!' → angry (expected angry)
  ✅ 'I'm scared of what might happen next.' → fear (expected fear)
  ❌ 'I'm a bit tired.' → fear (expected neutral)
  ✅ 'I'm can't wait for my birthday' → excited (expected excited)
  ❌ 'How long does it take to pick up up a piece o' → thinking (expected angry)
  ✅ 'tomato tomato' → neutral (expected neutral)


In [32]:
import shutil
from google.colab import files

# Zip the bert folder first since it's a directory
shutil.make_archive("emotion_bert_final", "zip", "./emotion_bert_final")

# Download both
files.download("emotion_bert_final.zip")
files.download("emotion_model.pkl")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>